# 03 - Monitoring & Drift Detection

Analyzes the transaction scoring log to detect data drift and defines the
retraining policy for this fraud detection model.

In [7]:
import pandas as pd
import numpy as np
import os

PROJECT_ROOT = os.path.dirname(os.getcwd())  # notebooks/ -> project root
score_log = pd.read_csv(os.path.join(PROJECT_ROOT, "monitoring", "score_log.csv"))
training_data = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "fraud_detection_clean.csv"))

print("Scored transactions logged so far:", len(score_log))
print(score_log.tail())


Scored transactions logged so far: 41
                     scored_at   amount merchant_category location  \
36  2026-09-04T13:11:30.989123  2613.53           grocery       UK   
37  2026-09-04T13:11:30.989123   867.37            gaming      USA   
38  2026-09-04T13:11:31.006125  1005.55       electronics    India   
39  2026-09-04T13:11:31.006125   760.35            travel       UK   
40  2026-09-04T13:11:31.017114  1782.46           grocery      UAE   

    fraud_probability  is_flagged  scoring_latency_ms  
36             0.0966        True                5.72  
37             0.0283       False                4.16  
38             0.0141       False                6.16  
39             0.0396       False                4.44  
40             0.0405       False                4.44  


In [8]:
def population_stability_index(expected, actual, bins=10):
    """PSI: <0.1 stable, 0.1-0.25 moderate drift, >0.25 significant drift."""
    breakpoints = np.percentile(expected, np.linspace(0, 100, bins + 1))
    breakpoints[0], breakpoints[-1] = -np.inf, np.inf

    expected_pct = np.histogram(expected, breakpoints)[0] / len(expected)
    actual_pct = np.histogram(actual, breakpoints)[0] / len(actual)

    expected_pct = np.where(expected_pct == 0, 0.0001, expected_pct)
    actual_pct = np.where(actual_pct == 0, 0.0001, actual_pct)

    return np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))


if len(score_log) < 30:
    print(f"Only {len(score_log)} transactions logged so far -- PSI needs a larger "
          f"sample (~30+) to be meaningful. Run score_transaction() a few more times "
          f"with varied sample data, then re-run this cell.")
else:
    psi_score = population_stability_index(training_data["amount"], score_log["amount"])
    print(f"PSI for 'amount': {psi_score:.4f}")
    if psi_score > 0.25:
        print("ALERT: Significant drift -- consider retraining.")
    elif psi_score > 0.1:
        print("WARNING: Moderate drift -- monitor closely.")
    else:
        print("Stable -- no action needed.")

PSI for 'amount': 1.6981
ALERT: Significant drift -- consider retraining.


In [10]:
import sys
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
from score_transaction import score_transaction

sample_transactions = training_data.sample(20, random_state=1).to_dict(orient="records")

for txn in sample_transactions:
    txn["timestamp"] = pd.Timestamp.now().isoformat()  # simulate a live transaction
    result = score_transaction(txn)

print("Logged 20 more transactions. Re-run the PSI cell above.")

def population_stability_index(expected, actual, bins=10):
    """PSI: <0.1 stable, 0.1-0.25 moderate drift, >0.25 significant drift."""
    breakpoints = np.percentile(expected, np.linspace(0, 100, bins + 1))
    breakpoints[0], breakpoints[-1] = -np.inf, np.inf

    expected_pct = np.histogram(expected, breakpoints)[0] / len(expected)
    actual_pct = np.histogram(actual, breakpoints)[0] / len(actual)

    expected_pct = np.where(expected_pct == 0, 0.0001, expected_pct)
    actual_pct = np.where(actual_pct == 0, 0.0001, actual_pct)

    return np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))


MIN_SAMPLES_FOR_PSI = 200  # below this, PSI is statistically unreliable

if len(score_log) < MIN_SAMPLES_FOR_PSI:
    print(f"Only {len(score_log)} transactions logged -- PSI needs at least "
          f"{MIN_SAMPLES_FOR_PSI}+ samples to be statistically reliable. "
          f"With small samples, empty bins inflate PSI artificially "
          f"(a well-documented limitation of the metric). "
          f"Showing an exploratory PSI below, but treat it as illustrative only.")
    # use fewer bins for a rough directional read on small samples
    exploratory_bins = max(2, min(5, len(score_log) // 5))
    psi_score = population_stability_index(
        training_data["amount"], score_log["amount"], bins=exploratory_bins
    )
    print(f"Exploratory PSI ({exploratory_bins} bins, not production-reliable): {psi_score:.4f}")
else:
    psi_score = population_stability_index(training_data["amount"], score_log["amount"])
    print(f"PSI for 'amount': {psi_score:.4f}")
    if psi_score > 0.25:
        print("ALERT: Significant drift -- consider retraining.")
    elif psi_score > 0.1:
        print("WARNING: Moderate drift -- monitor closely.")
    else:
        print("Stable -- no action needed.")

Logged 20 more transactions. Re-run the PSI cell above.
Only 41 transactions logged -- PSI needs at least 200+ samples to be statistically reliable. With small samples, empty bins inflate PSI artificially (a well-documented limitation of the metric). Showing an exploratory PSI below, but treat it as illustrative only.
Exploratory PSI (5 bins, not production-reliable): 0.2396


## Monitoring & Retraining Policy

**Track weekly:**
- PSI on `amount`, `transaction_frequency`, and `flag_sum`
- Flagged-transaction rate (% crossing the 0.05 threshold)
- Scoring latency (should stay well under 100ms)

**Track once true fraud labels arrive** (e.g. after chargebacks confirmed):
- Recompute precision/recall/PR-AUC on the new labeled batch
- Compare against baseline (Recall 0.70, Precision 0.09, PR-AUC 0.17)

**Retrain when:**
- PSI > 0.25 on any top-3 feature
- Recall on new labeled data drops >10 points below baseline
- 3 months since last training (calendar fallback)